In [ ]:
# =====================================================================
# CELL 1: NẠP DỮ LIỆU, KHỬ TRÙNG 2 TẦNG (SHA-256 + dHASH), TRỌNG SỐ ĐÁM ĐÔNG & CHIA STRATIFIED 70/15/15
# =====================================================================
import importlib
import os
import random
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler


def set_seed(seed=42):
    """Cố định hạt giống ngẫu nhiên (seed) để đảm bảo tính tái lập kết quả."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


KAGGLE_INPUT = Path("/kaggle/input")


def find_unique_file(filename):
    if Path(filename).exists():
        return Path(filename).resolve()
    matches = list(KAGGLE_INPUT.rglob(filename)) if KAGGLE_INPUT.exists() else []
    if len(matches) != 1:
        paths = "\n".join(str(path) for path in matches) or "(không tìm thấy)"
        raise FileNotFoundError(
            f"Cần tìm thấy đúng 1 file '{filename}', nhưng tìm thấy {len(matches)}:\n{paths}"
        )
    return matches[0]


DATA_UTILS = find_unique_file("data_utils.py")
PROJECT_DIR = DATA_UTILS.parent
IMG_CSV = find_unique_file("fer2013.csv")
LABEL_CSV = find_unique_file("fer2013new.csv")
SPLIT_MANIFEST_DIR = Path("split_manifest")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Thư mục mã nguồn:", PROJECT_DIR)
print("File ảnh:", IMG_CSV)
print("File nhãn:", LABEL_CSV)

import preprocessing
import data_utils
import dataset
import models
import eval_utils

importlib.reload(preprocessing)
importlib.reload(data_utils)
importlib.reload(dataset)
importlib.reload(models)
importlib.reload(eval_utils)

from preprocessing import PREPROCESSING_CONFIG, preprocess_fer_gray_48x48
from data_utils import load_fer2013_csv, load_split_from_manifest, compute_file_sha256, LABEL_NAMES
from dataset import FER2013Dataset, mixup_soft_batch
from models import CustomCNN
from eval_utils import (
    SoftWeightedKLDivLoss,
    ModelEMA,
    get_cosine_gamma,
    compute_smooth_class_weights,
    compute_effective_number_weights,
    compute_class_priors,
    apply_posthoc_logit_adjustment,
    optimize_classwise_posthoc_tau,
    compute_epoch_metrics,
    summarize_multi_seed_results,
    plot_learning_curves,
    denormalize_image,
    ten_crop_inference,
    generate_gradcam,
    evaluate_private_test,
)

# 1. Khử trùng 2 tầng (SHA-256 + cụm dHash có kiểm tra sai khác pixel) + Chia Stratified 70/15/15 theo khóa duy nhất y_hard
# Tắt retain_rare_ties_in_train=False và use_multilabel_stratify=False để đồng bộ nhất quán giữa 2 bước chia train/val/test.
set_seed(42)
(
    (X_train, y_train_soft, y_train_hard, w_train),
    (X_val, y_val_soft, y_val_hard, w_val),
    (X_test, y_test_soft, y_test_hard, w_test),
) = load_fer2013_csv(
    IMG_CSV,
    LABEL_CSV,
    return_soft_labels=True,
    return_sample_weights=True,
    use_stratify=True,
    use_multilabel_stratify=False,
    use_perceptual_dedup=True,
    near_dup_hamming=0,
    near_dup_pixel_mad_threshold=15.0,
    drop_ties=True,
    retain_rare_ties_in_train=False,
    rare_class_indices=(5, 6, 7),
    val_size=0.15,
    test_size=0.15,
    random_state=42,
    split_manifest_dir=SPLIT_MANIFEST_DIR,
    allow_positional_join=True,
)

EMOTION_DICT = {
    0: "Neutral",
    1: "Happiness",
    2: "Surprise",
    3: "Sadness",
    4: "Anger",
    5: "Disgust",
    6: "Fear",
    7: "Contempt",
}

In [ ]:
# =====================================================================
# CELL 2: HUẤN LUYỆN CUSTOM CNN (PARALLEL ECA+SE + RESIDUAL SPATIAL ATTENTION) + BALANCED LOGIT-ADJ
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Đang huấn luyện trên thiết bị:", device)

# Tính Class Priors pi_c từ phân phối Stratified sạch theo nhãn cứng y_train_hard
clean_class_counts = np.bincount(y_train_hard, minlength=8)
class_priors = compute_class_priors(clean_class_counts)

# Trọng số Smooth Inverse-Frequency (power=0.25, max_ratio=3.0)
smooth_class_weights = compute_smooth_class_weights(clean_class_counts, power=0.25, max_ratio=3.0)

print("Số lượng mẫu mỗi lớp trên tập Train sạch:", clean_class_counts.tolist())
print("Phân phối tiên nghiệm Stratified sạch (Class Priors pi_c):", np.round(class_priors, 4))
print("Trọng số Smooth Class Weights (power=0.25, max_ratio=3.0):", np.round(smooth_class_weights, 3))


def create_dataloaders(
    seed,
    batch_size=64,
    oversample_alpha=0.5,
    sampler_use_sample_weight=True,
):
    """
    oversample_alpha in [0.0, 1.0]:
      - 0.0: lấy mẫu ngẫu nhiên thuần tuý.
      - >0.0: oversampling nhẹ cho các lớp hiếm kết hợp trọng số mẫu.
    sampler_use_sample_weight (Mặc định True = hành vi hiện tại):
      - Khi True: sampler_weights = class_boost * w_train.
      - Khi False: sampler_weights = class_boost (tránh nhân sample_weight hai lần ở cả sampler và loss).
    """
    set_seed(seed)
    train_dataset = FER2013Dataset(
        X_train,
        y_train_soft,
        is_train=True,
        erasing_prob=0.15,
        erasing_scale=(0.01, 0.04),
        sample_weights=w_train,
        rare_classes=(5, 6, 7),
    )
    val_dataset = FER2013Dataset(X_val, y_val_soft, is_train=False)
    test_dataset = FER2013Dataset(X_test, y_test_hard, is_train=False)

    g = torch.Generator()
    g.manual_seed(seed)

    print(
        f"[DataLoader | Seed {seed}] oversample_alpha={oversample_alpha}, "
        f"sampler_use_sample_weight={sampler_use_sample_weight}"
    )

    if oversample_alpha > 0.0:
        class_boost = (1.0 - oversample_alpha) + oversample_alpha * smooth_class_weights[y_train_hard]
        if sampler_use_sample_weight:
            sampler_weights = class_boost * w_train
        else:
            sampler_weights = class_boost
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sampler_weights),
            num_samples=len(sampler_weights),
            replacement=True,
            generator=g,
        )
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            sampler=sampler,
            drop_last=True,
        )
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            generator=g,
            drop_last=True,
        )

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


def train_and_evaluate_custom_cnn(
    seed,
    num_epochs=85,
    ft_epochs=15,
    patience=25,
    mixup_alpha=0.2,
    mixup_prob=0.4,
    stage1_tau=0.30,
    stage1_max_gamma=0.5,
    gamma_warmup_epochs=15,
    posthoc_tau=0.10,
    use_ema=True,
    alpha_composite=0.5,
    sampler_use_sample_weight=True,
    log_val_before_after_tau=False,
):
    set_seed(seed)
    train_loader_p1, val_loader, test_loader = create_dataloaders(
        seed,
        batch_size=64,
        oversample_alpha=0.45,
        sampler_use_sample_weight=sampler_use_sample_weight,
    )

    model = CustomCNN(num_classes=8).to(device)
    model.freeze_backbone(False)
    ema = ModelEMA(model, decay=0.996) if use_ema else None

    criterion_p1 = SoftWeightedKLDivLoss(
        class_weights=None,
        class_priors=class_priors,
        logit_adj_tau=stage1_tau,
        gamma=0.0,
    ).to(device)

    optimizer = optim.SGD(
        model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4, nesterov=True
    )
    warmup_epochs = min(5, max(1, num_epochs // 10))
    scheduler_warmup = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs
    )
    scheduler_cosine = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, num_epochs - warmup_epochs), eta_min=1e-4
    )
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[scheduler_warmup, scheduler_cosine],
        milestones=[warmup_epochs],
    )

    ckpt_path = f"best_custom_cnn_seed{seed}.pth"
    best_val_composite = 0.0
    best_val_macro_f1 = 0.0
    patience_counter = 0

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": [],
        "train_macro_f1": [], "val_macro_f1": [],
        "val_weighted_f1": [], "val_composite": [],
    }

    print(
        f"\n>>> [CustomCNN | Seed {seed}] GIAI ĐOẠN 1: HỌC BIỂU DIỄN "
        f"({num_epochs} Epochs, Mixup={mixup_alpha}@p={mixup_prob}, stage1_tau={stage1_tau}, "
        f"val_posthoc_tau={posthoc_tau}, max_gamma={stage1_max_gamma})..."
    )
    for epoch in range(num_epochs):
        if hasattr(train_loader_p1.dataset, "set_erasing_enabled"):
            train_loader_p1.dataset.set_erasing_enabled(epoch < (num_epochs - 15))

        cur_gamma = get_cosine_gamma(
            epoch, num_epochs, max_gamma=stage1_max_gamma, warmup_epochs=gamma_warmup_epochs
        )
        criterion_p1.set_gamma(cur_gamma)

        model.train()
        running_loss, total_train = 0.0, 0
        train_preds, train_targets = [], []

        for batch in train_loader_p1:
            images, soft_labels, sample_weights = (
                batch[0].to(device),
                batch[1].to(device),
                batch[2].to(device),
            )
            mixed_images, mixed_labels, mixed_weights = mixup_soft_batch(
                images,
                soft_labels,
                sample_weights=sample_weights,
                alpha=mixup_alpha,
                p=mixup_prob,
                rare_classes=(5, 6, 7),
            )

            optimizer.zero_grad()
            outputs = model(mixed_images)
            loss = criterion_p1(outputs, mixed_labels, sample_weights=mixed_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            if ema is not None:
                ema.update(model)

            bs = images.size(0)
            running_loss += loss.item() * bs
            total_train += bs
            train_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            train_targets.extend(soft_labels.argmax(dim=1).cpu().numpy())

        train_loss = running_loss / total_train
        train_acc, train_macro_f1, _, _ = compute_epoch_metrics(train_targets, train_preds, alpha_composite)

        eval_net = ema.ema_model if ema is not None else model
        eval_net.eval()
        val_running_loss, total_val = 0.0, 0
        val_preds, val_targets = [], []

        with torch.no_grad():
            for images, soft_labels in val_loader:
                images, soft_labels = images.to(device), soft_labels.to(device)
                logits = eval_net(images)
                v_loss = criterion_p1(logits, soft_labels)
                adj_logits = apply_posthoc_logit_adjustment(logits, class_priors, tau=posthoc_tau)

                bs = images.size(0)
                val_running_loss += v_loss.item() * bs
                total_val += bs
                val_preds.extend(adj_logits.argmax(dim=1).cpu().numpy())
                val_targets.extend(soft_labels.argmax(dim=1).cpu().numpy())

        val_loss = val_running_loss / total_val
        val_acc, val_macro_f1, val_weighted_f1, val_composite = compute_epoch_metrics(
            val_targets, val_preds, alpha_composite
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_macro_f1"].append(train_macro_f1)
        history["val_macro_f1"].append(val_macro_f1)
        history["val_weighted_f1"].append(val_weighted_f1)
        history["val_composite"].append(val_composite)

        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        if val_composite > best_val_composite:
            best_val_composite = val_composite
            best_val_macro_f1 = val_macro_f1
            torch.save(eval_net.state_dict(), ckpt_path)
            patience_counter = 0
            flag = "-> Lưu best!"
        else:
            patience_counter += 1
            flag = ""

        if (epoch + 1) % 5 == 0 or flag:
            print(
                f"Epoch [{epoch+1:02d}/{num_epochs}] | LR: {current_lr:.5f} | Gamma: {cur_gamma:.2f} | "
                f"Loss: {train_loss:.4f}/{val_loss:.4f} | "
                f"Val Macro-F1: {val_macro_f1*100:.2f}% | Val Acc: {val_acc*100:.2f}% | "
                f"Val Comp: {val_composite*100:.2f}% {flag}"
            )

        if patience_counter >= patience:
            print(f"Dừng sớm (Early Stopping) Giai đoạn 1 tại Epoch {epoch + 1}!")
            break

    phase1_ran = len(history["train_loss"])

    # --- GIAI ĐOẠN 2: BLOCK4 + CLASSIFIER FINE-TUNING VỚI SMOOTH CLASS WEIGHTS ---
    print(f">>> [CustomCNN | Seed {seed}] GIAI ĐOẠN 2: BLOCK4 + CLASSIFIER FINE-TUNING ({ft_epochs} Epochs)...")
    best_state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(best_state)
    model.freeze_backbone(True, unfreeze_block4=True)

    train_loader_p2, _, _ = create_dataloaders(
        seed,
        batch_size=64,
        oversample_alpha=0.0,
        sampler_use_sample_weight=sampler_use_sample_weight,
    )
    if hasattr(train_loader_p2.dataset, "set_erasing_enabled"):
        train_loader_p2.dataset.set_erasing_enabled(False)

    if ema is not None:
        ema = ModelEMA(model, decay=0.99)

    criterion_p2 = SoftWeightedKLDivLoss(
        class_weights=smooth_class_weights,
        class_priors=class_priors,
        logit_adj_tau=stage1_tau,
        gamma=0.0,
    ).to(device)

    optimizer_ft = optim.SGD(
        [
            {"params": model.block4.parameters(), "lr": 3e-4},
            {"params": model.classifier.parameters(), "lr": 1e-3},
        ],
        momentum=0.9,
        weight_decay=1e-4,
        nesterov=True,
    )
    scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=ft_epochs, eta_min=1e-5)

    for epoch in range(ft_epochs):
        model.train()
        running_loss, total_train = 0.0, 0
        train_preds, train_targets = [], []

        for batch in train_loader_p2:
            images, soft_labels, sample_weights = (
                batch[0].to(device),
                batch[1].to(device),
                batch[2].to(device),
            )
            optimizer_ft.zero_grad()
            outputs = model(images)
            loss = criterion_p2(outputs, soft_labels, sample_weights=sample_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer_ft.step()

            if ema is not None:
                ema.update(model)

            bs = images.size(0)
            running_loss += loss.item() * bs
            total_train += bs
            train_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            train_targets.extend(soft_labels.argmax(dim=1).cpu().numpy())

        train_loss = running_loss / total_train
        train_acc, train_macro_f1, _, _ = compute_epoch_metrics(train_targets, train_preds, alpha_composite)

        eval_net = ema.ema_model if ema is not None else model
        eval_net.eval()
        val_running_loss, total_val = 0.0, 0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for images, soft_labels in val_loader:
                images, soft_labels = images.to(device), soft_labels.to(device)
                logits = eval_net(images)
                v_loss = criterion_p2(logits, soft_labels)
                adj_logits = apply_posthoc_logit_adjustment(logits, class_priors, tau=posthoc_tau)

                bs = images.size(0)
                val_running_loss += v_loss.item() * bs
                total_val += bs
                val_preds.extend(adj_logits.argmax(dim=1).cpu().numpy())
                val_targets.extend(soft_labels.argmax(dim=1).cpu().numpy())

        val_loss = val_running_loss / total_val
        val_acc, val_macro_f1, val_weighted_f1, val_composite = compute_epoch_metrics(
            val_targets, val_preds, alpha_composite
        )
        scheduler_ft.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_macro_f1"].append(train_macro_f1)
        history["val_macro_f1"].append(val_macro_f1)
        history["val_weighted_f1"].append(val_weighted_f1)
        history["val_composite"].append(val_composite)

        if val_composite > best_val_composite:
            best_val_composite = val_composite
            best_val_macro_f1 = val_macro_f1
            torch.save(eval_net.state_dict(), ckpt_path)
            print(
                f"FT Epoch [{epoch+1:02d}/{ft_epochs}] | Val Macro-F1: {val_macro_f1*100:.2f}% | "
                f"Val Acc: {val_acc*100:.2f}% | Val Comp: {val_composite*100:.2f}% -> Lưu best mới!"
            )

    # --- GIAI ĐOẠN 3: TỐI ƯU NGƯỠNG HẬU NGHIỆM CÓ RÀNG BUỘC CHẶT ---
    model.freeze_backbone(False)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()

    val_logits_list, val_targets_list = [], []
    with torch.no_grad():
        for images, soft_labels in val_loader:
            images = images.to(device)
            val_logits_list.append(model(images).cpu())
            val_targets_list.extend(soft_labels.argmax(dim=1).cpu().numpy())

    val_logits_all = torch.cat(val_logits_list, dim=0)

    # C3: Tùy chọn log kết quả Val trước khi tối ưu vector tau
    if log_val_before_after_tau:
        pre_adj_val = apply_posthoc_logit_adjustment(val_logits_all, class_priors, tau=posthoc_tau)
        pre_preds = pre_adj_val.argmax(dim=1).numpy()
        pre_acc, pre_mf1, pre_wf1, pre_comp = compute_epoch_metrics(
            val_targets_list, pre_preds, alpha_composite
        )

    best_tau_vec = optimize_classwise_posthoc_tau(
        val_logits_all,
        val_targets_list,
        class_priors,
        base_tau=posthoc_tau,
        max_acc_drop=0.005,
        alpha_composite=alpha_composite,
        rare_classes=(5, 6, 7),
        max_rare_recall_drop=0.02,
        l2_reg=0.05,
    )

    adj_val_logits = apply_posthoc_logit_adjustment(val_logits_all, class_priors, tau=best_tau_vec)
    val_preds_cal = adj_val_logits.argmax(dim=1).numpy()
    cal_val_acc, cal_val_macro_f1, cal_val_wf1, cal_val_comp = compute_epoch_metrics(
        val_targets_list, val_preds_cal, alpha_composite
    )

    if log_val_before_after_tau:
        print(
            f"\n[LOG C3 | KIỂM TRA MỨC ĐỘ OVERFIT VAL KHI TỐI ƯU TAU - SEED {seed}]\n"
            f"  • Trước tối ưu (tau={posthoc_tau:.2f}) -> Val Acc: {pre_acc*100:.2f}% | "
            f"Val Macro-F1: {pre_mf1*100:.2f}% | Val Comp: {pre_comp*100:.2f}%\n"
            f"  • Sau tối ưu  (best_tau_vec) -> Val Acc: {cal_val_acc*100:.2f}% | "
            f"Val Macro-F1: {cal_val_macro_f1*100:.2f}% | Val Comp: {cal_val_comp*100:.2f}% "
            f"(Delta Comp: {(cal_val_comp - pre_comp)*100:+.2f}%)"
        )

    best_val_composite = max(best_val_composite, cal_val_comp)
    best_val_macro_f1 = max(best_val_macro_f1, cal_val_macro_f1)

    # --- ĐÁNH GIÁ CUỐI CÙNG TRÊN TẬP TEST ---
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            logits = model(images)
            adj_logits = apply_posthoc_logit_adjustment(logits, class_priors, tau=best_tau_vec)
            preds = adj_logits.argmax(dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    metrics = evaluate_private_test(
        y_true, y_pred, EMOTION_DICT, title_prefix=f"CustomCNN (Seed {seed} - Balanced LogitAdj)"
    )
    metrics["best_val_composite"] = best_val_composite
    metrics["best_val_macro_f1"] = best_val_macro_f1
    metrics["posthoc_tau_vec"] = best_tau_vec
    return model, metrics, history, phase1_ran, ckpt_path

In [ ]:
# =====================================================================
# CELL 3: CHẠY 3 SEEDS & LƯU BEST.PTH + BEST_BUNDLE.PTH DỰA TRÊN TẬP VALIDATION (CHỐNG RÒ RỈ TẬP TEST)
# =====================================================================
if "class_priors" not in globals():
    class_priors = compute_class_priors(np.bincount(y_train_hard, minlength=8))

SEEDS = [42, 100, 2026]
custom_cnn_results = []
best_custom_model = None
best_val_score_across_seeds = -1.0
best_tau_vector = None

split_cfg_path = SPLIT_MANIFEST_DIR / "split_config.json"
split_config_hash = (
    compute_file_sha256(split_cfg_path) if split_cfg_path.exists() else "no_manifest_saved"
)

for seed in SEEDS:
    model_run, metrics_run, hist_run, p1_epochs, ckpt = train_and_evaluate_custom_cnn(
        seed=seed,
        num_epochs=85,
        ft_epochs=15,
        patience=25,
        mixup_alpha=0.2,
        mixup_prob=0.4,
        stage1_tau=0.30,
        stage1_max_gamma=0.5,
        gamma_warmup_epochs=15,
        posthoc_tau=0.10,
        use_ema=True,
        alpha_composite=0.5,
        sampler_use_sample_weight=True,
        log_val_before_after_tau=False,
    )
    custom_cnn_results.append(metrics_run)

    # Chỉ dùng điểm trên tập Validation (best_val_composite) để chọn mô hình lưu best.pth & best_bundle.pth
    if metrics_run["best_val_composite"] > best_val_score_across_seeds:
        best_val_score_across_seeds = metrics_run["best_val_composite"]
        best_custom_model = model_run
        best_tau_vector = metrics_run["posthoc_tau_vec"]

        # 1. Lưu best.pth và best_calibration.pt để giữ tương thích ngược
        torch.save(model_run.state_dict(), "best.pth")
        torch.save(
            {
                "class_priors": np.asarray(class_priors, dtype=np.float32).tolist(),
                "posthoc_tau": np.asarray(best_tau_vector, dtype=np.float32).tolist(),
            },
            "best_calibration.pt",
        )

        # 2. Lưu best_bundle.pth (chuyển toàn bộ numpy array sang list/tensor chuẩn để nạp bằng weights_only=True)
        bundle_dict = {
            "model_state_dict": model_run.state_dict(),
            "class_priors": np.asarray(class_priors, dtype=np.float32).tolist(),
            "posthoc_tau_vec": np.asarray(best_tau_vector, dtype=np.float32).tolist(),
            "use_eca": bool(model_run.block1.eca is not None),
            "use_spatial_attn": bool(model_run.block1.spatial_attn is not None),
            "label_names": list(LABEL_NAMES),
            "split_config_hash": str(split_config_hash),
            "preprocessing_config": dict(PREPROCESSING_CONFIG),
        }
        torch.save(bundle_dict, "best_bundle.pth")

        best_custom_history = (hist_run, p1_epochs)
        print(
            f"--> Đã cập nhật best.pth, best_calibration.pt & best_bundle.pth từ Seed {seed} "
            f"(Val Composite = {best_val_score_across_seeds*100:.2f}%)"
        )

summary_custom_cnn = summarize_multi_seed_results(
    custom_cnn_results,
    model_name="CustomCNN (3 Seeds - Parallel Attention + Clean Rare Ties + Balanced LogitAdj)",
)
plot_learning_curves(best_custom_history[0], phase1_epochs=best_custom_history[1])

In [ ]:
# =====================================================================
# CELL 4: SO SÁNH SINGLE-CROP VS TEN-CROP TRÊN VAL TRƯỚC -> ĐÁNH GIÁ TEST & GRAD-CAM
# =====================================================================
import matplotlib.pyplot as plt

best_custom_model.eval()


def run_single_and_tencrop_eval(X_data, y_hard_data, model, priors, tau_vec, batch_size=64):
    """Tính dự đoán cho cả Single-Crop (chuẩn Webcam) và Ten-Crop trên cùng tập dữ liệu."""
    ds = FER2013Dataset(X_data, y_hard_data, is_train=False)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    single_preds = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            logits = model(images)
            adj_logits = apply_posthoc_logit_adjustment(logits, priors, tau=tau_vec)
            single_preds.extend(adj_logits.argmax(dim=1).cpu().numpy())

    tencrop_preds = []
    for i in range(len(ds)):
        img_tensor, _ = ds[i]
        pred_tc, _ = ten_crop_inference(
            model,
            img_tensor,
            pad_size=4,
            crop_size=48,
            class_priors=priors,
            posthoc_tau=tau_vec,
        )
        tencrop_preds.append(pred_tc)

    return np.asarray(single_preds, dtype=np.int64), np.asarray(tencrop_preds, dtype=np.int64)


# 1. ĐÁNH GIÁ TRÊN TẬP VALIDATION TRƯỚC ĐỂ CHỌN GIAO THỨC SUY LUẬN
val_single_preds, val_tencrop_preds = run_single_and_tencrop_eval(
    X_val, y_val_hard, best_custom_model, class_priors, best_tau_vector
)
val_sc_acc, val_sc_mf1, val_sc_wf1, val_sc_comp = compute_epoch_metrics(y_val_hard, val_single_preds)
val_tc_acc, val_tc_mf1, val_tc_wf1, val_tc_comp = compute_epoch_metrics(y_val_hard, val_tencrop_preds)

print("=========================================================================")
print(f" BƯỚC 1: SO SÁNH SINGLE-CROP VS TEN-CROP TRÊN TẬP VALIDATION ({len(y_val_hard):,} MẪU)")
print("=========================================================================")
print(
    f"  • [VAL] Single-Crop : Composite={val_sc_comp*100:.2f}% | "
    f"Macro-F1={val_sc_mf1*100:.2f}% | Acc={val_sc_acc*100:.2f}% | Weighted-F1={val_sc_wf1*100:.2f}%"
)
print(
    f"  • [VAL] Ten-Crop    : Composite={val_tc_comp*100:.2f}% | "
    f"Macro-F1={val_tc_mf1*100:.2f}% | Acc={val_tc_acc*100:.2f}% | Weighted-F1={val_tc_wf1*100:.2f}%"
)

if val_tc_comp >= val_sc_comp:
    selected_protocol = "Ten-Crop"
    reference_protocol = "Single-Crop"
else:
    selected_protocol = "Single-Crop"
    reference_protocol = "Ten-Crop"

print(f"--> Ghi chú: Chọn giao thức theo Val: {selected_protocol} (Giao thức tham khảo: {reference_protocol})\n")

# 2. CHẠY TRÊN TẬP TEST VÀ IN RÕ KẾT QUẢ CHÍNH (THEO VAL) CÙNG KẾT QUẢ THAM KHẢO
test_single_preds, test_tencrop_preds = run_single_and_tencrop_eval(
    X_test, y_test_hard, best_custom_model, class_priors, best_tau_vector
)
test_sc_acc, test_sc_mf1, test_sc_wf1, test_sc_comp = compute_epoch_metrics(y_test_hard, test_single_preds)
test_tc_acc, test_tc_mf1, test_tc_wf1, test_tc_comp = compute_epoch_metrics(y_test_hard, test_tencrop_preds)

sc_role_label = (
    "KẾT QUẢ CHÍNH (giao thức chọn từ Val)"
    if selected_protocol == "Single-Crop"
    else "Tham khảo (khớp giao thức Webcam)"
)
tc_role_label = (
    "KẾT QUẢ CHÍNH (giao thức chọn từ Val)"
    if selected_protocol == "Ten-Crop"
    else "Tham khảo (Offline TTA)"
)

print("=========================================================================")
print(f" BƯỚC 2: KẾT QUẢ TRÊN TẬP TEST ({len(y_test_hard):,} MẪU) - GIAO THỨC CHÍNH: {selected_protocol.upper()}")
print("=========================================================================")
print(
    f"  • [TEST] Single-Crop [{sc_role_label}]: "
    f"Composite={test_sc_comp*100:.2f}% | Macro-F1={test_sc_mf1*100:.2f}% | "
    f"Acc={test_sc_acc*100:.2f}% | Weighted-F1={test_sc_wf1*100:.2f}%"
)
print(
    f"  • [TEST] Ten-Crop    [{tc_role_label}]: "
    f"Composite={test_tc_comp*100:.2f}% | Macro-F1={test_tc_mf1*100:.2f}% | "
    f"Acc={test_tc_acc*100:.2f}% | Weighted-F1={test_tc_wf1*100:.2f}%\n"
)

if selected_protocol == "Single-Crop":
    primary_preds, primary_title = (
        test_single_preds,
        "Test - KẾT QUẢ CHÍNH (giao thức chọn từ Val: Single-Crop, khớp Webcam)",
    )
    ref_preds, ref_title = (
        test_tencrop_preds,
        "Test - Tham khảo (Ten-Crop Offline TTA)",
    )
else:
    primary_preds, primary_title = (
        test_tencrop_preds,
        "Test - KẾT QUẢ CHÍNH (giao thức chọn từ Val: Ten-Crop)",
    )
    ref_preds, ref_title = (
        test_single_preds,
        "Test - Tham khảo (Single-Crop, khớp giao thức Webcam)",
    )

primary_metrics = evaluate_private_test(
    y_test_hard,
    primary_preds,
    EMOTION_DICT,
    title_prefix=primary_title,
)

reference_metrics = evaluate_private_test(
    y_test_hard,
    ref_preds,
    EMOTION_DICT,
    title_prefix=ref_title,
)

if selected_protocol == "Single-Crop":
    metrics_test_single = primary_metrics
    metrics_test_tencrop = reference_metrics
else:
    metrics_test_tencrop = primary_metrics
    metrics_test_single = reference_metrics

# 3. TRỰC QUAN HÓA BẢN ĐỒ NHIỆT GRAD-CAM TRÊN 8 LỚP CẢM XÚC
plt.figure(figsize=(16, 6))
test_ds_vis = FER2013Dataset(X_test, y_test_hard, is_train=False)

for cls_idx in range(8):
    cls_positions = np.where(y_test_hard == cls_idx)[0]
    if len(cls_positions) == 0:
        continue
    sample_pos = int(cls_positions[0])
    img_tensor, true_lbl = test_ds_vis[sample_pos]

    cam, pred_cls = generate_gradcam(best_custom_model, img_tensor, target_class=cls_idx)
    orig_img = denormalize_image(img_tensor)

    plt.subplot(2, 8, cls_idx + 1)
    plt.imshow(orig_img, cmap="gray")
    plt.title(f"Thực: {EMOTION_DICT[cls_idx]}", fontsize=10)
    plt.axis("off")

    plt.subplot(2, 8, cls_idx + 9)
    plt.imshow(orig_img, cmap="gray")
    plt.imshow(cam, cmap="jet", alpha=0.45)
    plt.title(f"Grad-CAM ({EMOTION_DICT[cls_idx]})", fontsize=9)
    plt.axis("off")

plt.suptitle("Trực quan hóa vùng chú ý Grad-CAM tại Block 4 (6x6) trên 8 lớp cảm xúc FER+", fontsize=14)
plt.tight_layout()
plt.show()